# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the [FAIR<sup>2</sup> dataset: Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values as defined by the Croissant schema.

We enumerate all record sets, fields, and columns, referencing entities by their `@id`.

In [ ]:
# List all available record sets and their fields using their `@id`
record_sets = list(dataset.record_sets)
print(f"Number of RecordSets: {len(record_sets)}")

for rs in record_sets:
    print(f"RecordSet @id: {rs.id}\n    Name: {rs.name}\n    Description: {rs.description}")
    print("    Field @id list:")
    for field in rs.fields:
        print(f"        - {field.id}  (name: {field.name}, type: {field.data_type})")
    print('---')

In [ ]:
# Print a sample record from each RecordSet by @id
for rs in record_sets:
    print(f"Sample record from RecordSet '@id': {rs.id}")
    for i, record in enumerate(dataset.records(record_set=rs.id)):
        print(record)
        if i >= 0:
            break
    print('---')

## 3. Data Extraction
Load data from one or more record set(s) into DataFrames for further analysis. Use the `@id` values for consistency as shown above.

Below, data is loaded for all record sets discovered in the schema.

In [ ]:
# Extract all records for each record set by @id into a DataFrame
dataframes = {}
record_set_ids = [rs.id for rs in record_sets]

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
    else:
        df = pd.DataFrame()
    dataframes[record_set_id] = df
    print(f"RecordSet: {record_set_id}, Columns: {df.columns.tolist() if len(df)>0 else 'No data'}")
    if not df.empty:
        display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common processing steps such as filtering, normalization, and grouping to one record set. We will work with the main record set if available (named or described as the primary protein data table).


In [ ]:
# Choose a record set with numeric content for EDA
# If you know the main table's @id (e.g. 'cr:recordSet/protein_table'), replace below accordingly, else pick the first.
if len(record_set_ids) > 0:
    main_record_set_id = record_set_ids[0]
    df = dataframes[main_record_set_id]
    print(f"Working with RecordSet @id = {main_record_set_id}")
    print(f"Columns: {df.columns.tolist()}")
else:
    print("No record sets found.")

# Choose a numeric field by inspecting column types or names; suppose 'coverage' or 'cr:field/coverage' as an example (adjust appropriately)
numeric_candidates = [col for col in df.columns if df[col].dtype.kind in 'fi']
print(f"Candidate numeric fields: {numeric_candidates}")

if len(numeric_candidates) > 0:
    numeric_field = numeric_candidates[0]
    threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
    # Example: filter rows greater than threshold
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a likely categorical or string/text field
    group_candidates = [col for col in df.columns if df[col].dtype == 'O' and col != numeric_field]
    if len(group_candidates) > 0:
        group_field = group_candidates[0]
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field} (showing means):")
        display(grouped_df.head())
    else:
        print("No groupable fields found.")
else:
    print("No numeric fields found for EDA.")

## 5. Visualization
Visualize data distributions such as histogram for a numeric field and boxplots by group if possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(numeric_candidates) > 0:
    # Histogram of the numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    # Boxplot by group if possible
    if len(group_candidates) > 0:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load a Croissant-described dataset using `mlcroissant`, explore its schema, extract data by `@id`, and perform basic exploratory data analysis and visualization.

- All dataset entities were referenced by their unique Croissant `@id` for reproducibility.
- The dataset metadata, record set structure, field names and data types were reviewed programmatically.
- Simple numeric filtering, normalization, grouping, and visualizations were shown. Further scientific analysis can be conducted with the processed DataFrames.

For further questions, consult the [FAIR<sup>2</sup> Croissant Schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) and the [`mlcroissant` documentation](https://mlcommons.github.io/croissant/api/python/mlcroissant.html).
